# Trénink sítě na přestavovače brzdy (Výkaz vozů)

Z fotek, které vozmistři pořídili v aplikaci **Výkaz vozů**, natrénuje malou neuronovou síť (MobileNetV3) pro každý přestavovač:

| druh | polohy |
|---|---|
| `rezim` | G / P |
| `naklad` | prázdný / ložený / samočinný |
| `kohout` | brzda zapnuta / vypnuta |
| `rucni` | ruční brzda odbrzděna / zabrzděna / nemá |
| `blocks` | druh brzdových špalíků K / L / LL / D / F |

**Postup**
1. V aplikaci: *Data pro trénink → Exportovat nové*, ZIP uložit na Google Disk do složky `Vykaz_vozu_data`.
2. Tady: *Runtime → Change runtime type → T4 GPU* (jde to i bez GPU, jen pomaleji) a *Run all*.
3. Hotové sítě se uloží na Disk do `Vykaz_vozu_modely/` – soubory `*.onnx` a `brzdy.json` zkopírovat do složky `models/` aplikace a vydat novou verzi.

Síť se natrénuje jen u přestavovače, který má aspoň dvě polohy s dostatkem fotek (`MIN_FOTEK`).

Kontrolní fotky jsou z vozů, které síť při tréninku neuvidí. Přesnost je vyvážená (průměr po polohách) a porovná se s tím,
kolik by šlo poznat jen podle jasu fotky – když síť nevede o víc než 10 bodů, učí se spíš den a noc než polohu páky.
Výpis se uloží i na Disk do `Vykaz_vozu_modely/trenink_<razítko>.txt`.
Na začátek stačí desítky fotek od každé polohy, spolehlivá síť chce stovky – z různých vozů, za dne i v noci.

In [ ]:
!pip -q install onnx onnxscript onnxruntime

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DISK = Path('/content/drive/MyDrive')
VSTUP = DISK / 'Vykaz_vozu_data'      # sem ukládat ZIPy z aplikace
VYSTUP = DISK / 'Vykaz_vozu_modely'   # sem se zapíšou hotové sítě
DATA = Path('/content/data')

MIN_FOTEK = 8      # nejméně fotek od každé polohy, jinak se přestavovač přeskočí
EPOCHY = 20
NA_VSECH = True    # po kontrole se výsledná síť natrénuje znovu na všech fotkách (stejně EPOCHY, poslední stav)
VELIKOST = 224     # aplikace zmenšuje celou fotku na čtverec 224 × 224 (bez ořezu) – musí sedět
VYSTUP.mkdir(exist_ok=True)
print('ZIPy:', sorted(p.name for p in VSTUP.glob('*.zip')))

In [ ]:
# Rozbalení všech exportů do jedné složky (stejná fotka ve více exportech se jen přepíše).
import zipfile, csv, io, collections
DATA.mkdir(exist_ok=True)
for z in sorted(VSTUP.glob('*.zip')):
    with zipfile.ZipFile(z) as f:
        f.extractall(DATA)

# Sloupec s hodnotou se jmenoval „poloha“, od verze 2 aplikace „hodnota“.
def poloha(r):
    return r.get('hodnota') or r.get('poloha') or ''

def smazana(r):
    return r.get('smazano') == '1'

# Popisky bere z labels.csv posledního exportu, kde fotka byla (opravy polohy se tak projeví).
# Fotka vyřazená v aplikaci (koš v detailu snímku) má v novějším exportu řádek se smazano = 1 – vynechá se.
POPISKY = {}
for z in sorted(VSTUP.glob('*.zip')):
    with zipfile.ZipFile(z) as f:
        text = f.read('labels.csv').decode('utf-8-sig')
    for r in csv.DictReader(io.StringIO(text), delimiter=';'):
        POPISKY[r['soubor'].split('/')[-1]] = r

DRUHY = ['rezim', 'naklad', 'kohout', 'rucni', 'blocks']
for druh in DRUHY:
    pocty = collections.Counter(poloha(r) for r in POPISKY.values() if r['druh'] == druh and not smazana(r))
    print(f'{druh:8}', dict(pocty) or '–')

In [ ]:
import random, time, json
import numpy as np
import torch, torch.nn as nn
from PIL import Image
from torchvision import models, transforms

zarizeni = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Počítá se na:', zarizeni)

NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
# Bez zrcadlení: strana, na které páka je, určuje polohu.
# Jas široce: polohy nejsou rovnoměrně ve dne a v noci (G a vypnutá brzda skoro jen v noci s bleskem),
# síť se nemá naučit den / noc místo polohy páky.
TRENINK = transforms.Compose([
    transforms.Resize((VELIKOST, VELIKOST)),
    transforms.RandomAffine(degrees=12, translate=(0.06, 0.06), scale=(0.85, 1.15)),
    transforms.ColorJitter((0.35, 1.8), 0.5, 0.4, 0.03),
    transforms.RandomGrayscale(0.15),
    transforms.RandomApply([transforms.GaussianBlur(5)], p=0.2),
    transforms.ToTensor(), NORM,
])
KONTROLA = transforms.Compose([transforms.Resize((VELIKOST, VELIKOST)), transforms.ToTensor(), NORM])

class Fotky(torch.utils.data.Dataset):
    def __init__(self, polozky, tf):
        self.polozky, self.tf = polozky, tf
    def __len__(self):
        return len(self.polozky)
    def __getitem__(self, i):
        cesta, trida = self.polozky[i][:2]
        return self.tf(Image.open(cesta).convert('RGB')), trida

def fotky_druhu(druh):
    # aktuální poloha z labels.csv; soubor leží ve složce podle polohy v době exportu
    # → {poloha: [(cesta, vůz)]}, fotka bez čísla vozu je „vůz“ sama pro sebe
    polozky = collections.defaultdict(list)
    for jmeno, r in POPISKY.items():
        if r['druh'] != druh or smazana(r):
            continue
        nalezene = list((DATA / druh).glob(f'*/{jmeno}'))
        if nalezene:
            polozky[poloha(r)].append((nalezene[0], r.get('vuz') or jmeno))
    return polozky

def jas(cesta):
    # průměrný jas fotky 0–255
    return float(np.asarray(Image.open(cesta).convert('L').resize((16, 16))).mean())

def vyvazena(odhady, skutecne):
    # průměr přesností po polohách – síť, která říká vždy P, má 50 %, ne podíl fotek P
    po = {t: np.mean([o == t for o, s in zip(odhady, skutecne) if s == t]) for t in sorted(set(skutecne))}
    return float(np.mean(list(po.values()))), po

def jen_podle_jasu(trenink, kontrola, n):
    # kolik kontrolních fotek by poznal jen jas fotky (den / noc), naučený na trénovacích fotkách:
    # u dvou poloh nejlepší práh, u víc poloh nejbližší průměrný jas polohy
    jt = [(jas(c), t) for c, t, _ in trenink]
    jk = [(jas(c), t) for c, t, _ in kontrola]
    if n == 2:
        nejlepsi = max(((p, dole) for p in range(0, 256, 2) for dole in (0, 1)),
                       key=lambda pd: vyvazena([pd[1] if j < pd[0] else 1 - pd[1] for j, _ in jt], [t for _, t in jt])[0])
        odhad = lambda j: nejlepsi[1] if j < nejlepsi[0] else 1 - nejlepsi[1]
    else:
        prumer = [np.mean([j for j, t in jt if t == k]) for k in range(n)]
        odhad = lambda j: int(np.argmin([abs(j - p) for p in prumer]))
    return vyvazena([odhad(j) for j, _ in jk], [t for _, t in jk])[0]

# výpis se uloží i na Disk (Vykaz_vozu_modely/trenink_<razítko>.txt), ať jde přečíst bez Colabu
VYPIS = []
def rekni(*co):
    radek = ' '.join(str(x) for x in co)
    print(radek)
    VYPIS.append(radek)

In [ ]:
def rozdel(polozky, tridy):
    # kontrolní fotky po vozech: fotky jednoho vozu jsou jen v tréninku, nebo jen v kontrole
    # (jinak síť pozná vůz, ne polohu), od každé polohy aspoň 20 % fotek, nejméně 2
    random.seed(1)
    kontrolni = set()
    for t in tridy:
        vozy = sorted({v for _, v in polozky[t]})
        random.shuffle(vozy)
        chci = max(2, len(polozky[t]) // 5)
        for v in vozy:
            if sum(1 for _, w in polozky[t] if w in kontrolni) >= chci:
                break
            kontrolni.add(v)
    trenink, kontrola = [], []
    for t in tridy:
        for c, v in sorted(polozky[t]):
            (kontrola if v in kontrolni else trenink).append((c, tridy.index(t), v))
    return trenink, kontrola

def uc(trenink, tridy, kontrola=None):
    # s kontrolou se vybere epocha s nejlepší vyváženou přesností, bez ní zůstane síť po poslední epoše
    sit = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    sit.classifier[3] = nn.Linear(sit.classifier[3].in_features, len(tridy))
    sit = sit.to(zarizeni)
    # vyrovnání nestejného počtu fotek mezi polohami
    pocty = np.bincount([t for _, t, _ in trenink], minlength=len(tridy))
    vahy = torch.tensor(len(trenink) / (len(tridy) * np.maximum(pocty, 1)), dtype=torch.float32, device=zarizeni)
    ztrata = nn.CrossEntropyLoss(weight=vahy, label_smoothing=0.05)
    opt = torch.optim.AdamW(sit.parameters(), lr=5e-4, weight_decay=1e-4)
    plan = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHY)
    dl_t = torch.utils.data.DataLoader(Fotky(trenink, TRENINK), batch_size=32, shuffle=True, num_workers=2)
    dl_k = torch.utils.data.DataLoader(Fotky(kontrola, KONTROLA), batch_size=64, num_workers=2) if kontrola else None
    skutecne = [t for _, t, _ in kontrola or []]

    nejlepsi, stav = -1, None
    for ep in range(EPOCHY):
        sit.train()
        for x, y in dl_t:
            opt.zero_grad()
            ztrata(sit(x.to(zarizeni)), y.to(zarizeni)).backward()
            opt.step()
        plan.step()
        if not kontrola:
            continue
        sit.eval()
        odhady = []
        with torch.no_grad():
            for x, _ in dl_k:
                odhady += sit(x.to(zarizeni)).argmax(1).cpu().tolist()
        presnost, po = vyvazena(odhady, skutecne)
        if presnost >= nejlepsi:
            nejlepsi, stav = presnost, {k: v.detach().clone() for k, v in sit.state_dict().items()}
        rekni(f'  epocha {ep + 1:2}: vyvážená přesnost {presnost:.0%} (' +
              ', '.join(f'{tridy[t]} {p:.0%}' for t, p in po.items()) + ')')
    if stav:
        sit.load_state_dict(stav)
    sit.eval()
    return sit, nejlepsi

def trenuj(druh):
    polozky = {k: v for k, v in fotky_druhu(druh).items() if len(v) >= MIN_FOTEK}
    if len(polozky) < 2:
        rekni(f'{druh}: málo fotek, přeskakuji')
        return None
    tridy = sorted(polozky)
    trenink, kontrola = rozdel(polozky, tridy)
    rekni(f'{druh}: třídy {tridy}, trénink {len(trenink)}, kontrola {len(kontrola)} '
          f'({len({v for *_, v in kontrola})} vozů, které síť při tréninku neuvidí)')

    sit, nejlepsi = uc(trenink, tridy, kontrola)
    skutecne = [t for _, t, _ in kontrola]

    # kontrolní fotky po polohách a chybně poznané – často odhalí špatně zadanou polohu
    odhady, jistoty = [], []
    with torch.no_grad():
        for cesta, t, vuz in kontrola:
            p = sit(KONTROLA(Image.open(cesta).convert('RGB'))[None].to(zarizeni)).softmax(1)[0]
            odhady.append(int(p.argmax()))
            jistoty.append(float(p.max()))
            if odhady[-1] != t:
                rekni(f'  ✗ {cesta.name} (vůz {vuz}, jas {jas(cesta):.0f}): zadáno {tridy[t]}, '
                      f'síť říká {tridy[odhady[-1]]} na {jistoty[-1]:.0%}')
    for t in range(len(tridy)):
        dobre = sum(1 for o, s in zip(odhady, skutecne) if s == t and o == t)
        rekni(f'  {tridy[t]}: {dobre} z {skutecne.count(t)} kontrolních fotek')
    jisto = [o == s for o, s, j in zip(odhady, skutecne, jistoty) if j >= 0.9]
    rekni(f'  s jistotou ≥ 90 %: {len(jisto)} z {len(kontrola)} fotek, z nich správně {sum(jisto)}')
    jen_jas = jen_podle_jasu(trenink, kontrola, len(tridy))
    rekni(f'{druh}: vyvážená přesnost sítě {nejlepsi:.0%}, jen podle jasu fotky (den / noc) {jen_jas:.0%}'
          + ('  ← síť nepoznává o moc víc než den a noc' if nejlepsi < jen_jas + 0.1 else ''))
    if NA_VSECH:
        # kontrola výše ověřila postup; výsledná síť se učí ze všech fotek (víc příkladů, hlavně vzácných poloh)
        rekni(f'  výsledná síť: znovu na všech {len(trenink) + len(kontrola)} fotkách, {EPOCHY} epoch')
        sit, _ = uc(trenink + kontrola, tridy)
    return sit.cpu(), tridy, nejlepsi, jen_jas

In [ ]:
import onnxruntime as ort

VYPIS.clear()
razitko = time.strftime('%Y%m%d_%H%M')
konfig_cesta = VYSTUP / 'brzdy.json'
konfig = json.loads(konfig_cesta.read_text()) if konfig_cesta.exists() else {}

for druh in DRUHY:
    vysledek = trenuj(druh)
    if not vysledek:
        continue
    sit, tridy, presnost, jen_jas = vysledek
    soubor = f'{druh}_{razitko}.onnx'   # nové jméno = aplikace si síť určitě stáhne znovu
    vzor = torch.randn(1, 3, VELIKOST, VELIKOST)
    try:
        torch.onnx.export(sit, vzor, VYSTUP / soubor, input_names=['obraz'], output_names=['logity'],
                          opset_version=17, dynamo=False)
    except TypeError:   # starší PyTorch nezná parametr dynamo
        torch.onnx.export(sit, vzor, VYSTUP / soubor, input_names=['obraz'], output_names=['logity'], opset_version=17)
    # kontrola: ONNX musí počítat stejně jako PyTorch
    sez = ort.InferenceSession(str(VYSTUP / soubor))
    rozdil = np.abs(sez.run(None, {'obraz': vzor.numpy()})[0] - sit(vzor).detach().numpy()).max()
    rekni(f'{soubor}: rozdíl ONNX × PyTorch {rozdil:.2e}')
    # accuracy = vyvážená přesnost na kontrolních vozech, brightness_only = totéž jen podle jasu fotky
    konfig[druh] = {'file': soubor, 'classes': tridy, 'size': VELIKOST,
                    'accuracy': round(float(presnost), 3), 'brightness_only': round(float(jen_jas), 3),
                    'trained': razitko}

konfig_cesta.write_text(json.dumps(konfig, ensure_ascii=False, indent=2))
(VYSTUP / f'trenink_{razitko}.txt').write_text('\n'.join(VYPIS) + '\n')
print(json.dumps(konfig, ensure_ascii=False, indent=2))
print('Do složky models/ aplikace zkopírujte brzdy.json a nové soubory .onnx z', VYSTUP)